# Stable baselines

In [12]:
import numpy as np
from stable_baselines3.common.evaluation import evaluate_policy


def measure_performance(model, env=None, n_episodes=20, deterministic=True):
    """
    Measures performance using an existing model and environment instance.
    :param model: The trained SB3 model instance (e.g., DQN, PPO, etc.)
    :param env: (Optional) A specific environment to evaluate on.
                If None, uses model.get_env().
    :param n_episodes: Number of episodes to evaluate.
    :param deterministic: Whether to use deterministic actions (recommended for eval).
    """
    # 1. Determine which environment to use
    eval_env = env if env is not None else model.get_env()
    if eval_env is None:
        print("Error: No environment provided and model has no environment attached.")
        return None
    print(f"--- Evaluating Performance ({n_episodes} episodes) ---")
    # 2. Run the evaluation
    # This returns a list of rewards and episode lengths
    episode_rewards, episode_lengths = evaluate_policy(
        model,
        eval_env,
        n_eval_episodes=n_episodes,
        return_episode_rewards=True,
        deterministic=deterministic,
    )
    # 3. Calculate metrics
    mean_reward = np.mean(episode_rewards)
    std_reward = np.std(episode_rewards)
    mean_len = np.mean(episode_lengths)
    # Example success threshold (adjust based on your specific task)
    success_rate = (np.array(episode_rewards) > 0).mean() * 100
    # 4. Print results
    print(f"Mean Reward:       {mean_reward:.2f} +/- {std_reward:.2f}")
    print(f"Avg Episode Length: {mean_len:.1f} steps")
    print(f"Success Rate:      {success_rate:.1f}%")
    return {
        "mean_reward": mean_reward,
        "std_reward": std_reward,
        "mean_length": mean_len,
        "all_rewards": episode_rewards,
    }

## Cart pole

In [17]:
from stable_baselines3 import DQN
from stable_baselines3.common.env_util import make_vec_env

# Parallel environments
vec_env = make_vec_env("CartPole-v1", n_envs=4)

model = DQN("MlpPolicy", vec_env, verbose=0, tensorboard_log="./dqn_tensorboard/")

In [28]:
# 2. Training Loop
total_iterations = 1
steps_per_iteration = 100_000
for i in range(total_iterations):
    print(f"\n--- Iteration {i+1}/{total_iterations} ---")
    # CRITICAL: reset_num_timesteps=False
    # This ensures the internal step counter and learning rate
    # continue from where they left off instead of restarting at 0.
    model.learn(total_timesteps=steps_per_iteration, reset_num_timesteps=False)
    model.save("dqn_cartpole")

    # 3. Evaluate the current version
    # Note: evaluate_policy (inside measure_performance) resets the env
    stats = measure_performance(model, n_episodes=10)
    # Optional: Save checkpoint if performance is good
    # if stats['mean_reward'] > best_reward:


--- Iteration 1/1 ---
--- Evaluating Performance (10 episodes) ---
Mean Reward:       500.00 +/- 0.00
Avg Episode Length: 500.0 steps
Success Rate:      100.0%


In [ ]:
obs = vec_env.reset()
while True:
    action, _states = model.predict(obs)
    obs, rewards, dones, info = vec_env.step(action)
    vec_env.render("human")

KeyboardInterrupt: 

: 

In [ ]:
from stable_baselines3 import A2C
from stable_baselines3.common.env_util import make_vec_env

# Parallel environments
vec_env = make_vec_env("CartPole-v1", n_envs=4)

model = A2C("MlpPolicy", vec_env, verbose=1)
model.learn(total_timesteps=25000)
model.save("a2c_cartpole")

del model  # remove to demonstrate saving and loading

model = A2C.load("a2c_cartpole")

obs = vec_env.reset()
while True:
    action, _states = model.predict(obs)
    obs, rewards, dones, info = vec_env.step(action)
    vec_env.render("human")

Using cpu device
------------------------------------
| rollout/              |          |
|    ep_len_mean        | 29.3     |
|    ep_rew_mean        | 29.3     |
| time/                 |          |
|    fps                | 4338     |
|    iterations         | 100      |
|    time_elapsed       | 0        |
|    total_timesteps    | 2000     |
| train/                |          |
|    entropy_loss       | -0.64    |
|    explained_variance | 0.0574   |
|    learning_rate      | 0.0007   |
|    n_updates          | 99       |
|    policy_loss        | 1.64     |
|    value_loss         | 7.78     |
------------------------------------
------------------------------------
| rollout/              |          |
|    ep_len_mean        | 41.4     |
|    ep_rew_mean        | 41.4     |
| time/                 |          |
|    fps                | 5114     |
|    iterations         | 200      |
|    time_elapsed       | 0        |
|    total_timesteps    | 4000     |
| train/             

objc[45819]: Class SDLApplication is implemented in both /Users/rich/Developer/Github/VariousDataAnalysis/reinforcement_learning/q_learning/.venv/lib/python3.12/site-packages/cv2/.dylibs/libSDL2-2.0.0.dylib (0x1615f0890) and /Users/rich/Developer/Github/VariousDataAnalysis/reinforcement_learning/q_learning/.venv/lib/python3.12/site-packages/pygame/.dylibs/libSDL2-2.0.0.dylib (0x165ead2c8). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.
objc[45819]: Class SDLAppDelegate is implemented in both /Users/rich/Developer/Github/VariousDataAnalysis/reinforcement_learning/q_learning/.venv/lib/python3.12/site-packages/cv2/.dylibs/libSDL2-2.0.0.dylib (0x1615f08e0) and /Users/rich/Developer/Github/VariousDataAnalysis/reinforcement_learning/q_learning/.venv/lib/python3.12/site-packages/pygame/.dylibs/libSDL2-2.0.0.dylib (0x165ead318). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be remo

KeyboardInterrupt: 

In [8]:
dir(model)

['__abstractmethods__',
 '__annotations__',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__slots__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_abc_impl',
 '_current_progress_remaining',
 '_custom_logger',
 '_dump_logs',
 '_episode_num',
 '_excluded_save_params',
 '_get_policy_from_name',
 '_get_torch_save_params',
 '_init_callback',
 '_last_episode_starts',
 '_last_obs',
 '_last_original_obs',
 '_logger',
 '_maybe_recommend_cpu',
 '_n_updates',
 '_num_timesteps_at_start',
 '_setup_learn',
 '_setup_lr_schedule',
 '_setup_model',
 '_stats_window_size',
 '_total_timesteps',
 '_update_current_progress_remaining',
 '_update_info_buffer',
 '_update_learning_rate',
 '_vec_normalize_env',
 '_wr

In [13]:
model.policy

ActorCriticPolicy(
  (features_extractor): FlattenExtractor(
    (flatten): Flatten(start_dim=1, end_dim=-1)
  )
  (pi_features_extractor): FlattenExtractor(
    (flatten): Flatten(start_dim=1, end_dim=-1)
  )
  (vf_features_extractor): FlattenExtractor(
    (flatten): Flatten(start_dim=1, end_dim=-1)
  )
  (mlp_extractor): MlpExtractor(
    (policy_net): Sequential(
      (0): Linear(in_features=4, out_features=64, bias=True)
      (1): Tanh()
      (2): Linear(in_features=64, out_features=64, bias=True)
      (3): Tanh()
    )
    (value_net): Sequential(
      (0): Linear(in_features=4, out_features=64, bias=True)
      (1): Tanh()
      (2): Linear(in_features=64, out_features=64, bias=True)
      (3): Tanh()
    )
  )
  (action_net): Linear(in_features=64, out_features=2, bias=True)
  (value_net): Linear(in_features=64, out_features=1, bias=True)
)

In [ ]:
from stable_baselines3 import A2C
from stable_baselines3.common.env_util import make_vec_env

# Parallel environments
vec_env = make_vec_env("CartPole-v1", n_envs=4)

model = A2C("MlpPolicy", vec_env, verbose=1)
model.learn(total_timesteps=25000)
model.save("a2c_cartpole")

# del model # remove to demonstrate saving and loading

Using cpu device
------------------------------------
| rollout/              |          |
|    ep_len_mean        | 22.3     |
|    ep_rew_mean        | 22.3     |
| time/                 |          |
|    fps                | 6295     |
|    iterations         | 100      |
|    time_elapsed       | 0        |
|    total_timesteps    | 2000     |
| train/                |          |
|    entropy_loss       | -0.592   |
|    explained_variance | -0.00992 |
|    learning_rate      | 0.0007   |
|    n_updates          | 99       |
|    policy_loss        | 0.0171   |
|    value_loss         | 23.1     |
------------------------------------
------------------------------------
| rollout/              |          |
|    ep_len_mean        | 24.5     |
|    ep_rew_mean        | 24.5     |
| time/                 |          |
|    fps                | 7014     |
|    iterations         | 200      |
|    time_elapsed       | 0        |
|    total_timesteps    | 4000     |
| train/             

In [6]:
model.action_space

Discrete(2)

In [7]:
# model = A2C.load("a2c_cartpole")

obs = vec_env.reset()
while True:
    action, _states = model.predict(obs)
    obs, rewards, dones, info = vec_env.step(action)
    vec_env.render("human")

KeyboardInterrupt: 

## Lunar lander

In [ ]:
import gymnasium as gym

from stable_baselines3 import DQN
from stable_baselines3.common.evaluation import evaluate_policy

# Create environment
env = gym.make("LunarLander-v3", render_mode="rgb_array")

env = gym.make("ALE/Breakout-v5", render_mode="human")


# Instantiate the agent
model = DQN("MlpPolicy", env, verbose=1)
# Train the agent and display a progress bar
model.learn(total_timesteps=int(2e5), progress_bar=True)
# Save the agent
model.save("dqn_lunar")
del model  # delete trained model to demonstrate loading

# Load the trained agent
# NOTE: if you have loading issue, you can pass `print_system_info=True`
# to compare the system on which the model was trained vs the current one
# model = DQN.load("dqn_lunar", env=env, print_system_info=True)
model = DQN.load("dqn_lunar", env=env)

# Evaluate the agent
# NOTE: If you use wrappers with your environment that modify rewards,
#       this will be reflected here. To evaluate with original rewards,
#       wrap environment in a "Monitor" wrapper before other wrappers.
mean_reward, std_reward = evaluate_policy(model, model.get_env(), n_eval_episodes=10)

# Enjoy trained agent
vec_env = model.get_env()
obs = vec_env.reset()
for i in range(1000):
    action, _states = model.predict(obs, deterministic=True)
    obs, rewards, dones, info = vec_env.step(action)
    vec_env.render("human")

/Users/rich/Developer/Github/VariousDataAnalysis/reinforcement_learning/q_learning/.venv/lib/python3.12/site-packag
es/rich/live.py:260: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 88       |
|    ep_rew_mean      | -197     |
|    exploration_rate | 0.983    |
| time/               |          |
|    episodes         | 4        |
|    fps              | 1085     |
|    time_elapsed     | 0        |
|    total_timesteps  | 352      |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 0.886    |
|    n_updates        | 62       |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 95.4     |
|    ep_rew_mean      | -206     |
|    exploration_rate | 0.964    |
| time/               |          |
|    episodes         | 8        |
|    fps              | 1453     |
|    time_elapsed     | 0        |
|    total_timesteps  | 763      |
| train/              |        

Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


## Breakout

In [29]:
import gymnasium as gym
import ale_py
from stable_baselines3 import DQN
from stable_baselines3.common.callbacks import CheckpointCallback
from stable_baselines3.common.evaluation import evaluate_policy
import os

# --- CONFIGURATION ---
ENV_ID = "ALE/Breakout-v5"
SAVE_DIR = "./logs/dqn_checkpoints/"
TOTAL_STEPS = 200000
SAVE_FREQ = 50000  # Save a version every 50,000 steps
# Create directory if it doesn't exist
os.makedirs(SAVE_DIR, exist_ok=True)
# 1. Setup Environment
# Note: For Atari games like Breakout, CnnPolicy is usually better than MlpPolicy
env = gym.make(ENV_ID, render_mode="human")
# --- PHASE 1: TRAINING WITH CHECKPOINTS ---
# This callback will save files like: dqn_model_50000_steps.zip, dqn_model_100000_steps.zip
checkpoint_callback = CheckpointCallback(
    save_freq=SAVE_FREQ, save_path=SAVE_DIR, name_prefix="dqn_model"
)
print(f"Starting training for {TOTAL_STEPS} steps...")
model = DQN("CnnPolicy", env, verbose=1)  # Changed to CnnPolicy as Breakout uses pixels
model.learn(
    total_timesteps=TOTAL_STEPS, callback=checkpoint_callback, progress_bar=True
)
model.save(os.path.join(SAVE_DIR, "dqn_model_final"))
print("Training complete. Checkpoints saved in:", SAVE_DIR)

Starting training for 200000 steps...

Using cpu device

Wrapping the env with a `Monitor` wrapper

Wrapping the env in a DummyVecEnv.

Wrapping the env in a VecTransposeImage.

/Users/rich/Developer/Github/VariousDataAnalysis/reinforcement_learning/q_learning/.venv/lib/python3.12/site-packag
es/stable_baselines3/common/buffers.py:242: UserWarning: This system does not have apparently enough memory to 
store the complete replay buffer 201.62GB > 1.35GB
  warnings.warn(

----------------------------------
| rollout/            |          |
|    ep_len_mean      | 175      |
|    ep_rew_mean      | 1        |
|    exploration_rate | 0.967    |
| time/               |          |
|    episodes         | 4        |
|    fps              | 12       |
|    time_elapsed     | 58       |
|    total_timesteps  | 701      |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 2.54e-05 |
|    n_updates        | 150      |
----------------------------------

----------------------------------
| rollout/            |          |
|    ep_len_mean      | 173      |
|    ep_rew_mean      | 1        |
|    exploration_rate | 0.934    |
| time/               |          |
|    episodes         | 8        |
|    fps              | 11       |
|    time_elapsed     | 121      |
|    total_timesteps  | 1386     |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 7.52e-05 |
|    n_updates        | 321      |
----------------------------------

KeyboardInterrupt: 

In [ ]:
model.replay_buffer.

array([[[[[0, 0, 0, ..., 0, 0, 0],
          [0, 0, 0, ..., 0, 0, 0],
          [0, 0, 0, ..., 0, 0, 0],
          ...,
          [0, 0, 0, ..., 0, 0, 0],
          [0, 0, 0, ..., 0, 0, 0],
          [0, 0, 0, ..., 0, 0, 0]],

         [[0, 0, 0, ..., 0, 0, 0],
          [0, 0, 0, ..., 0, 0, 0],
          [0, 0, 0, ..., 0, 0, 0],
          ...,
          [0, 0, 0, ..., 0, 0, 0],
          [0, 0, 0, ..., 0, 0, 0],
          [0, 0, 0, ..., 0, 0, 0]],

         [[0, 0, 0, ..., 0, 0, 0],
          [0, 0, 0, ..., 0, 0, 0],
          [0, 0, 0, ..., 0, 0, 0],
          ...,
          [0, 0, 0, ..., 0, 0, 0],
          [0, 0, 0, ..., 0, 0, 0],
          [0, 0, 0, ..., 0, 0, 0]]]],



       [[[[0, 0, 0, ..., 0, 0, 0],
          [0, 0, 0, ..., 0, 0, 0],
          [0, 0, 0, ..., 0, 0, 0],
          ...,
          [0, 0, 0, ..., 0, 0, 0],
          [0, 0, 0, ..., 0, 0, 0],
          [0, 0, 0, ..., 0, 0, 0]],

         [[0, 0, 0, ..., 0, 0, 0],
          [0, 0, 0, ..., 0, 0, 0],
          [0, 0, 0

In [ ]:
# --- PHASE 2: LOADING A SPECIFIC VERSION ---
# Change this string to the specific step count you want to test
VERSION_TO_LOAD = "dqn_model_100000_steps.zip"
load_path = os.path.join(SAVE_DIR, VERSION_TO_LOAD)
if os.path.exists(load_path):
    print(f"Loading version: {VERSION_TO_LOAD}")
    # Load the specific checkpoint
    model = DQN.load(load_path, env=env)
    # Evaluate the specific version
    mean_reward, std_reward = evaluate_policy(model, model.get_env(), n_eval_episodes=5)
    print(f"Mean reward for {VERSION_TO_LOAD}: {mean_reward:.2f} +/- {std_reward:.2f}")
    # --- SIMULATE ---
    print("Starting simulation...")
    vec_env = model.get_env()
    obs = vec_env.reset()
    for i in range(1000):
        action, _states = model.predict(obs, deterministic=True)
        obs, rewards, dones, info = vec_env.step(action)
        vec_env.render("human")
else:
    print(f"Error: Could not find checkpoint at {load_path}")

In [28]:
import gymnasium as gym

from stable_baselines3 import DQN
from stable_baselines3.common.evaluation import evaluate_policy

import ale_py

gym.register_envs(ale_py)

# Create environment
env = gym.make("ALE/Breakout-v5", render_mode="human")


# Instantiate the agent
model = DQN("MlpPolicy", env, verbose=1)
# Train the agent and display a progress bar
model.learn(total_timesteps=int(2e5), progress_bar=True)
# Save the agent
model.save("dqn_breakout")
del model  # delete trained model to demonstrate loading

# Load the trained agent
# NOTE: if you have loading issue, you can pass `print_system_info=True`
# to compare the system on which the model was trained vs the current one
# model = DQN.load("dqn_lunar", env=env, print_system_info=True)
model = DQN.load("dqn_breakout", env=env)

# Evaluate the agent
# NOTE: If you use wrappers with your environment that modify rewards,
#       this will be reflected here. To evaluate with original rewards,
#       wrap environment in a "Monitor" wrapper before other wrappers.
mean_reward, std_reward = evaluate_policy(model, model.get_env(), n_eval_episodes=10)

# Enjoy trained agent
vec_env = model.get_env()
obs = vec_env.reset()
for i in range(1000):
    action, _states = model.predict(obs, deterministic=True)
    obs, rewards, dones, info = vec_env.step(action)
    vec_env.render("human")

A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]
objc[45819]: Class SDL_RumbleMotor is implemented in both /Users/rich/Developer/Github/VariousDataAnalysis/reinforcement_learning/q_learning/.venv/lib/python3.12/site-packages/cv2/.dylibs/libSDL2-2.0.0.dylib (0x1615f0d40) and /Users/rich/Developer/Github/VariousDataAnalysis/reinforcement_learning/q_learning/.venv/lib/python3.12/site-packages/ale_py/libSDL2-2.0.0.dylib (0x17de90910). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.


/Users/rich/Developer/Github/VariousDataAnalysis/reinforcement_learning/q_learning/.venv/lib/python3.12/site-packag
es/rich/live.py:260: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

objc[45819]: Class SDL_RumbleContext is implemented in both /Users/rich/Developer/Github/VariousDataAnalysis/reinforcement_learning/q_learning/.venv/lib/python3.12/site-packages/cv2/.dylibs/libSDL2-2.0.0.dylib (0x1615f0d90) and /Users/rich/Developer/Github/VariousDataAnalysis/reinforcement_learning/q_learning/.venv/lib/python3.12/site-packages/ale_py/libSDL2-2.0.0.dylib (0x17de90960). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.
objc[45819]: Class SDLApplication is implemented in both /Users/rich/Developer/Github/VariousDataAnalysis/reinforcement_learning/q_learning/.venv/lib/python3.12/site-packages/cv2/.dylibs/libSDL2-2.0.0.dylib (0x1615f0890) and /Users/rich/Developer/Github/VariousDataAnalysis/reinforcement_learning/q_learning/.venv/lib/python3.12/site-packages/ale_py/libSDL2-2.0.0.dylib (0x17de909b0). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or rename

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Wrapping the env in a VecTransposeImage.
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 174      |
|    ep_rew_mean      | 1        |
|    exploration_rate | 0.967    |
| time/               |          |
|    episodes         | 4        |
|    fps              | 14       |
|    time_elapsed     | 47       |
|    total_timesteps  | 697      |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 0.000649 |
|    n_updates        | 149      |
----------------------------------


KeyboardInterrupt: 

In [25]:
rewards

array([-0.12929237], dtype=float32)

In [23]:
model.get_env()

In [24]:
env

<TimeLimit<OrderEnforcing<PassiveEnvChecker<LunarLander<LunarLander-v3>>>>>

In [17]:
# Enjoy trained agent
vec_env = model.get_env()
obs = vec_env.reset()
for i in range(1000):
    action, _states = model.predict(obs, deterministic=True)
    obs, rewards, dones, info = vec_env.step(action)
    vec_env.render("human")

In [22]:
obs

array([[ 0.00149927,  1.4190471 ,  0.15185723,  0.36119175, -0.00173061,
        -0.03439793,  0.        ,  0.        ]], dtype=float32)

In [ ]:
vec_env.

array([0])